# 로또 6/45 당첨번호 통계 분석

동행복권 API로 수집한 전회차 데이터를 바탕으로:
- 번호별 출현 빈도 (핫/콜드 번호)
- 연속번호·홀짝 분포
- 합계·구간 분포
- 번호 간 동반 출현 히트맵

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'matplotlib', 'seaborn', 'requests'])

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('..') / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from crawl_lotto import crawl

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA = Path('..') / 'data' / 'lotto.csv'

## 1. 데이터 수집

In [ ]:
if DATA.exists():
    df = pd.read_csv(DATA)
    print(f'기존 데이터 로드: {len(df)}회차 ({df["date"].min()} ~ {df["date"].max()})')
else:
    print('데이터 수집 중... (약 10분 소요)')
    df = crawl(start=1)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {len(df)}행')

df.head()

## 2. 번호별 출현 빈도 (핫/콜드)

In [ ]:
num_cols = ['n1', 'n2', 'n3', 'n4', 'n5', 'n6']
all_nums = df[num_cols].values.flatten()
freq = pd.Series(all_nums).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#e74c3c' if f >= freq.quantile(0.75) else
          '#3498db' if f <= freq.quantile(0.25) else '#95a5a6'
          for f in freq]
ax.bar(freq.index, freq.values, color=colors, width=0.8)
ax.set_xlabel('번호')
ax.set_ylabel('출현 횟수')
ax.set_title('로또 6/45 번호별 출현 빈도 (빨강=상위25% 핫, 파랑=하위25% 콜드)')
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
plt.tight_layout()
plt.show()

print('\n상위 10 핫번호:', freq.nlargest(10).index.tolist())
print('하위 10 콜드번호:', freq.nsmallest(10).index.tolist())

## 3. 합계 분포

In [ ]:
df['sum'] = df[num_cols].sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['sum'], bins=40, color='#2ecc71', edgecolor='white')
ax.axvline(df['sum'].mean(), color='red', linestyle='--', label=f'평균: {df["sum"].mean():.1f}')
ax.set_xlabel('6개 번호 합계')
ax.set_ylabel('회차 수')
ax.set_title('당첨번호 합계 분포')
ax.legend()
plt.tight_layout()
plt.show()

print(f'합계 범위: {df["sum"].min()} ~ {df["sum"].max()}')
print(f'평균: {df["sum"].mean():.1f}, 중앙값: {df["sum"].median()}')

## 4. 홀짝 비율 분포

In [ ]:
df['odd_count'] = df[num_cols].apply(lambda r: sum(v % 2 == 1 for v in r), axis=1)
odd_dist = df['odd_count'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(odd_dist.index, odd_dist.values, color='#9b59b6')
ax.set_xlabel('홀수 개수')
ax.set_ylabel('회차 수')
ax.set_title('당첨번호 홀짝 분포')
for i, v in zip(odd_dist.index, odd_dist.values):
    ax.text(i, v + 1, f'{v/len(df)*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 5. 번호 간 동반 출현 히트맵

In [ ]:
co_matrix = np.zeros((45, 45), dtype=int)

for _, row in df[num_cols].iterrows():
    nums = sorted(row.values)
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            a, b = nums[i] - 1, nums[j] - 1
            co_matrix[a][b] += 1
            co_matrix[b][a] += 1

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(co_matrix, cmap='YlOrRd', ax=ax,
            xticklabels=range(1, 46), yticklabels=range(1, 46))
ax.set_title('번호 간 동반 출현 횟수')
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

# 상위 5쌍
pairs = []
for i in range(45):
    for j in range(i+1, 45):
        pairs.append(((i+1, j+1), co_matrix[i][j]))
pairs.sort(key=lambda x: -x[1])
print('가장 많이 함께 나온 번호 쌍 TOP 10:')
for p, cnt in pairs[:10]:
    print(f'  {p[0]:2d} & {p[1]:2d} → {cnt}회')

## 6. 구간별 분포 (1~10, 11~20, 21~30, 31~40, 41~45)

In [ ]:
def zone(n):
    if n <= 10: return '1-10'
    if n <= 20: return '11-20'
    if n <= 30: return '21-30'
    if n <= 40: return '31-40'
    return '41-45'

zone_counts = pd.Series(all_nums).apply(zone).value_counts().reindex(
    ['1-10', '11-20', '21-30', '31-40', '41-45'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(zone_counts.index, zone_counts.values, color='#1abc9c')
ax.set_xlabel('구간')
ax.set_ylabel('출현 횟수')
ax.set_title('번호 구간별 출현 빈도')
for i, v in enumerate(zone_counts.values):
    ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.show()